In [1]:
import numpy as np
import pandas as pd

In [2]:
rng = np.random.default_rng(42)

df_sampling = pd.DataFrame(
    {
        "user_id": np.arange(1, 101),
        "category": rng.choice(["A", "B", "C"], size=100, p=[0.5, 0.3, 0.2]),
        "score": rng.normal(loc=75, scale=10, size=100).round(2),
    }
)

In [3]:
rng.choice(["A", "B", "C"])

np.str_('B')

In [4]:
df_sampling.head()

,user_id,category,score
0,1,B,79.00
1,2,A,65.95
2,3,C,71.22
3,4,B,87.99
4,5,A,71.44


In [5]:
df_sampling.shape

(100, 3)

In [6]:
df_sampling['score'].describe()[['mean', 'std']]

mean    74.859800
std      9.843752
Name: score, dtype: float64

In [7]:
df_sampling['category'].value_counts()

category
A    53
B    33
C    14
Name: count, dtype: int64

In [8]:
df_sampling['category'].value_counts(normalize=True)

category
A    0.53
B    0.33
C    0.14
Name: proportion, dtype: float64

Random Sampling

In [9]:
rs = df_sampling.sample(n = 30, random_state=42)

In [10]:
rs['score'].describe()[['mean', 'std']]

mean    72.739667
std      8.956177
Name: score, dtype: float64

In [11]:
rs.head()

,user_id,category,score
83,84,A,77.19
53,54,B,64.64
70,71,A,73.43
45,46,C,77.68
44,45,C,53.68


In [12]:
rs.groupby('category')['score'].describe()[['mean', 'std', 'count']]

,mean,std,count
category,,,
A,73.318000,10.134300,15.0
B,74.975556,6.108523,9.0
C,67.940000,8.961924,6.0


In [13]:
rs['category'].value_counts(normalize=True)

category
A    0.5
B    0.3
C    0.2
Name: proportion, dtype: float64

Stratified Sampling

In [14]:
stratified_sampling = (df_sampling
                       .groupby('category', group_keys=False)
                       .sample(frac=0.2, random_state=42))

stratified_sampling.head()

,user_id,category,score
38,39,A,78.14
83,84,A,77.19
92,93,A,92.24
28,29,A,89.63
85,86,A,86.06


In [15]:
(stratified_sampling
.groupby('category')['score']
.describe()[['mean', 'std', 'count']])

,mean,std,count
category,,,
A,76.688182,10.510287,11.0
B,78.715714,10.081969,7.0
C,63.650000,5.447054,3.0


In [16]:
(df_sampling
.groupby('category')['score']
.describe()[['mean', 'std', 'count']])

,mean,std,count
category,,,
A,74.200000,8.973076,53.0
B,76.272424,10.014058,33.0
C,74.027857,12.705514,14.0


In [17]:
print(stratified_sampling.shape)
stratified_sampling['category'].value_counts(normalize=True)

(21, 3)


category
A    0.523810
B    0.333333
C    0.142857
Name: proportion, dtype: float64

In [18]:
print(df_sampling.shape)
df_sampling['category'].value_counts(normalize=True)

(100, 3)


category
A    0.53
B    0.33
C    0.14
Name: proportion, dtype: float64

In [19]:
stratified_sampling = (df_sampling
                       .groupby('category')
                       .sample(n = 12))

In [20]:
stratified_sampling['category'].value_counts()

category
A    12
B    12
C    12
Name: count, dtype: int64

In [21]:
df_sampling.head()

,user_id,category,score
0,1,B,79.00
1,2,A,65.95
2,3,C,71.22
3,4,B,87.99
4,5,A,71.44


In [22]:
def control_treatment(df:pd.DataFrame, p:list)->pd.DataFrame:
    rng = np.random.default_rng(42)
    df_sampling_splitting = df.assign(
    group = rng.choice(['control', 'treatment'], size = len(df_sampling), p = p)
)

    return df_sampling_splitting

In [23]:
df_sampling_splitting = control_treatment(df_sampling, [0.5, 0.5])

In [24]:
df_sampling_splitting[['group', 'category']].value_counts()

group      category
control    A           53
treatment  B           33
           C           14
Name: count, dtype: int64

In [25]:
df_stratified_split = (
    df_sampling
    .assign(
        group=lambda x: (
            x.groupby("category")["user_id"]
            .transform(
                lambda g: rng.permutation(
                    ["control"] * (len(g)//2) + 
                    ["treatment"] * (len(g) - len(g)//2)
                )
            )
        )
    )
)

df_stratified_split.head()

,user_id,category,score,group
0,1,B,79.00,control
1,2,A,65.95,control
2,3,C,71.22,control
3,4,B,87.99,control
4,5,A,71.44,control


In [26]:
df_stratified_split[['group', 'category']].value_counts()

group      category
treatment  A           27
control    A           26
treatment  B           17
control    B           16
           C            7
treatment  C            7
Name: count, dtype: int64

In [29]:
#pip install statsmodels scipy

  Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl (9.5 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   ----

In [30]:
import numpy as np
import pandas as pd
import math
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
import scipy
import matplotlib.pyplot as plt

- effect_size
- power
- alpha
- sample size

In [31]:
N = TTestIndPower().solve_power(effect_size = 0.4, power = 0.8,
                            alpha = 0.05)

N

99.08032514676704

In [38]:
N = TTestIndPower().solve_power(effect_size = 0.4, power = 0.8,
                            nobs1 = 100)

N

0.048508871922401664

In [37]:
TTestIndPower().solve_power(effect_size = 0.4, power = 0.8,
                            alpha = 0.05)

99.08032514676704

In [40]:
N = TTestIndPower().solve_power(effect_size = 0.365, power = 0.8,
                            alpha = 0.05, alternative = 'larger')

N = int(N)
N

93

In [41]:
expr = pd.read_csv('../data/ab_testing/experiment.csv')
expr.head()

,user_id,viewing_time,Group
0,4b5630ee914e848e8d07221556b0a2fb,38.354937,control
1,c01f179e4b57ab8bd9de309e6d576c48,49.534278,control
2,11946e7a3ed5e1776e81c0f0ecd383d0,35.468325,control
3,234a2a5581872457b9fe1187d1616b13,69.014875,control
4,dd4ad37ee474732a009111e3456e7ed7,51.547207,control


In [42]:
expr.groupby('Group')['viewing_time'].mean()

Group
control      48.386186
treatment    52.081302
Name: viewing_time, dtype: float64

- T-test
- X1 and X2

In [43]:
control = expr[expr['Group'] == 'control']['viewing_time']
treatment = expr[expr['Group'] == 'treatment']['viewing_time']

In [70]:
t_test_results = ttest_ind(treatment,control)
t_value, p_value = t_test_results

In [49]:
f"T-Statistics: {t_value:.4F}"

'T-Statistics: 1.6002'

In [52]:
f"P-Value: {p_value:.4F}"

'P-Value: 0.1128'

In [55]:
diff=treatment.mean() - control.mean()
sd_pooled=math.sqrt((treatment.std()**2+ control.std()**2)/2)

In [56]:
f"The detected Effect: {diff/sd_pooled:.4F}"

'The detected Effect: 0.3200'

Practical Significance

In [57]:
def measures(data):
    desc = data.describe()
    x1 = desc.loc['mean', 'score1']
    x2 = desc.loc['mean', 'score2']
    s1 = desc.loc['std', 'score1']
    s2 = desc.loc['std', 'score2']
    return x1, x2, s1, s2

In [58]:
def ttest(x1,x2,s1,s2,n):
    t_value = (x1-x2)/math.sqrt(s1**2/n+s2**2/n)
    p_value = scipy.stats.t.sf(abs(t_value), df=n-1)*2
    return f't-value: {t_value:.4f}', f'p-value: {p_value:.4f}'

Case 1

In [59]:
case1=pd.read_csv("../data/ab_testing/case1.csv")
case1.head()

,score1,score2
0,85,87
1,85,86
2,86,87
3,86,86
4,85,86


In [60]:
case1.describe().loc[['count','mean','std']]

,score1,score2
count,20.000000,20.000000
mean,85.550000,86.400000
std,0.510418,0.502625


In [61]:
# t-test for case 1
c1=measures(case1)
c1

(np.float64(85.55),
 np.float64(86.4),
 np.float64(0.5104177855340404),
 np.float64(0.5026246899500346))

In [62]:
ttest(*c1,20)

('t-value: -5.3065', 'p-value: 0.0000')

Case 2

In [64]:
case2=pd.read_csv("../data/ab_testing/case2.csv")
case2.head()

c2 = measures(case2)
c2

(np.float64(90.65),
 np.float64(90.75),
 np.float64(2.777257261172764),
 np.float64(2.7886046312580213))

In [65]:
ttest(*c2,20)

('t-value: -0.1136', 'p-value: 0.9107')

In [66]:
ttest(*c2,200)

('t-value: -0.3593', 'p-value: 0.7197')

In [67]:
ttest(*c2,2000)

('t-value: -1.1363', 'p-value: 0.2560')

In [68]:
ttest(*c2,20000)

('t-value: -3.5933', 'p-value: 0.0003')